In [ ]:
!pip install pandas requests beautifulsoup4 numpy langdetect fasttext matplotlib seaborn lxml spacy spacy-langdetect

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import csv
import json
import pandas as pd
import glob
import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
from langdetect import detect, DetectorFactory
from urllib.parse import urlparse, urlunparse
import string
import spacy
from spacy.language import Language
from spacy_langdetect import LanguageDetector
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import FuncFormatter

In [ ]:
# load the file
working_csv_path = "./clean_df_no_dup_risks2.csv"
df = pd.read_csv(working_csv_path)

In [7]:
# number of rows and columns
print("Cleaned DataFrame shape:", df.shape)

# Identify duplicated risks text
duplicated_risks = df[df.duplicated(subset=['risks_html'], keep=False)]
print(f"\nNumber of projects with duplicated 'risks_html': {duplicated_risks.shape[0]}")
print(duplicated_risks[['slug', 'risks_html']])

# number of unique values in risks_html with state 'failed'
num_unique_risks_failed = df[df['state'] == 'failed']['risks_html'].nunique()
print(f"Number of unique values in 'risks_html' for failed projects: {num_unique_risks_failed}")

# number of unique values in risks_html with state 'successful'
num_unique_risks_successful = df[df['state'] == 'successful']['risks_html'].nunique()
print(f"Number of unique values in 'risks_html' for successful projects: {num_unique_risks_successful}")

Cleaned DataFrame shape: (44995, 53)

Number of projects with duplicated 'risks_html': 321
                                                    slug  \
44674  191563098/jane-brown-a-story-of-courage-surviv...   
44675  canadianinvasion/canadian-invasions-choose-you...   
44676          90895326/student-operated-fabrication-lab   
44677     mikemoreno/mike-moreno-group-fall-2011-release   
44678  870238516/sci-fi-janitors-vs-the-redbull-soapb...   
...                                                  ...   
44990            1356242277/kill-pretty-magazine-issue-2   
44991    1946588017/rock-n-roller-remote-controller-ep-5   
44992        sarahtracey/sarah-tracey-saves-the-universe   
44993  112268952/help-the-jackie-myers-band-make-thei...   
44994  federicomaiozzi/a-tripoli-italian-expansionism...   

                                              risks_html  
44674  <div class="mobile-show py3"></div><h2 class="...  
44675  <div class="mobile-show py3"></div><h2 class="...  
44676  <div

In [ ]:
# check number of unique story_html values
unique_story_html = df['story_html'].nunique()
print(f"Number of unique values in 'story_html': {unique_story_html}")


Number of unique values in 'story_html': 44994


In [ ]:
# rows with duplicate story_html
duplicated_story = df[df.duplicated(subset=['story_html'], keep=False)]
print(f"\nNumber of projects with duplicated 'story_html': {duplicated_story.shape[0]}")
pd.set_option('display.max_colwidth', None)
print(duplicated_story[['slug', 'story_html']])


Number of projects with duplicated 'story_html': 2
                                     slug  \
21629  destinyaa/destiny-alright-already1   
34993   destinyaa/destiny-alright-already   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              story_html  
21629  <div><p>I want to raise funds to print a multitude of copies to give away to women's shelters/prisons for encouragement. Eventually, then 

In [6]:
# print urls_web_project for index 34993 and 21629
print("\nURLs for specific indices:")
print("Index 34993 URL:", df.loc[34993, 'urls_web_project'])
print("Index 21629 URL:", df.loc[21629, 'urls_web_project'])


URLs for specific indices:
Index 34993 URL: https://www.kickstarter.com/projects/destinyaa/destiny-alright-already
Index 21629 URL: https://www.kickstarter.com/projects/destinyaa/destiny-alright-already1


In [ ]:
# check unique rewards_html values
unique_rewards_html = df['rewards_html'].nunique()
print(f"Number of unique values in 'rewards_html': {unique_rewards_html}")

Number of unique values in 'rewards_html': 44995


In [9]:
# rows with duplicate rewards_html
duplicated_rewards = df[df.duplicated(subset=['rewards_html'], keep=False)]
print(f"\nNumber of projects with duplicated 'rewards_html': {duplicated_rewards.shape[0]}")
print(duplicated_rewards[['slug', 'rewards_html']])


Number of projects with duplicated 'rewards_html': 0
Empty DataFrame
Columns: [slug, rewards_html]
Index: []


Pre-processing

In [8]:
# print columns
print("\nColumns in df:") 
print(df.columns)
print("Number of columns:", len(df.columns))


Columns in df:
Index(['blurb', 'category_name', 'category_parent_name', 'country',
       'created_at', 'creator_has_admin_message_badge',
       'creator_ppo_has_action', 'creator_backing_action_count', 'currency',
       'deadline', 'disable_communication', 'goal', 'id', 'is_disliked',
       'is_liked', 'is_starrable', 'launched_at', 'location_name',
       'location_country', 'location_state', 'location_type', 'name',
       'photo_1536x864', 'prelaunch_activated', 'profile_id',
       'profile_project_id', 'profile_name', 'profile_blurb',
       'profile_show_feature_image',
       'profile_should_show_feature_image_section',
       'profile_feature_image_attributes_image_urls_default',
       'profile_feature_image_attributes_image_urls_baseball_card', 'slug',
       'source_url', 'staff_pick', 'static_usd_rate', 'urls_web_project',
       'urls_web_rewards', 'video', 'story_html', 'risks_html', 'rewards_html',
       'pledged', 'backers_count', 'usd_pledged', 'converted_pledged

In [9]:
# check missing values
print(df.isnull().sum())

blurb                                                            0
category_name                                                    0
category_parent_name                                           798
country                                                          0
created_at                                                       0
creator_has_admin_message_badge                                  0
creator_ppo_has_action                                           0
creator_backing_action_count                                     0
currency                                                         0
deadline                                                         0
disable_communication                                            0
goal                                                             0
id                                                               0
is_disliked                                                      0
is_liked                                                      

In [10]:
# delete profile_name, profile_blurb, is_in_post_campaign_pledging_phase
df = df.drop(columns=['profile_name', 'profile_blurb', 'is_in_post_campaign_pledging_phase'], errors='ignore')

In [ ]:
# unique values in category_parent_name
unique_parent_categories = df['category_parent_name'].unique()
print(f"Unique parent categories: {unique_parent_categories}")

Unique parent categories: <StringArray>
[  'Publishing',         'Food',        'Games',   'Technology',
       'Comics',      'Fashion', 'Film & Video',       'Crafts',
  'Photography',       'Design',        'Music',          'Art',
      'Theater',   'Journalism',            nan,        'Dance']
Length: 16, dtype: str


In [12]:
#unique values in category_name
unique_categories = df['category_name'].unique()
print(f"Unique categories: {unique_categories}")

Unique categories: <StringArray>
[        'Poetry',    'Restaurants',          'Zines',  'Playing Cards',
      'Wearables',     'Live Games',    'Comic Books',  'Childrenswear',
 'Tabletop Games',      'Webseries',
 ...
          'Glass',        'Romance',     'Technology',         'Quilts',
         'Design',     'Publishing',      'Taxidermy',         'Comics',
        'Fashion',       'Chiptune']
Length: 161, dtype: str


In [ ]:
# check missing values in category_parent_name
missing_parents = df[df['category_parent_name'].isna()]
missing_parents['category_name'].value_counts()

category_name
Dance           200
Crafts          170
Journalism      105
Theater          88
Food             60
Photography      52
Music            50
Art              33
Technology       10
Film & Video      8
Design            7
Comics            6
Publishing        4
Games             3
Fashion           2
Name: count, dtype: int64

In [14]:
# Fill missing parent names with the category itself because they are already parent categories
df['category_parent_name'] = df['category_parent_name'].fillna(
    df['category_name']
)

In [15]:
# fill missing values in location_country, location_state
df['location_name'] = df['location_name'].fillna('Unknown')
df['location_state'] = df['location_state'].fillna('Unknown')
df['location_country'] = df['location_country'].fillna('Unknown')
df['location_type'] = df['location_type'].fillna('Unknown')

In [16]:
# check missing values again
print(df.isnull().sum())

blurb                                                        0
category_name                                                0
category_parent_name                                         0
country                                                      0
created_at                                                   0
creator_has_admin_message_badge                              0
creator_ppo_has_action                                       0
creator_backing_action_count                                 0
currency                                                     0
deadline                                                     0
disable_communication                                        0
goal                                                         0
id                                                           0
is_disliked                                                  0
is_liked                                                     0
is_starrable                                           

In [17]:
# number of columns
print("Number of columns:", len(df.columns))

# columns
print("Columns:")
print(df.columns)

Number of columns: 50
Columns:
Index(['blurb', 'category_name', 'category_parent_name', 'country',
       'created_at', 'creator_has_admin_message_badge',
       'creator_ppo_has_action', 'creator_backing_action_count', 'currency',
       'deadline', 'disable_communication', 'goal', 'id', 'is_disliked',
       'is_liked', 'is_starrable', 'launched_at', 'location_name',
       'location_country', 'location_state', 'location_type', 'name',
       'photo_1536x864', 'prelaunch_activated', 'profile_id',
       'profile_project_id', 'profile_show_feature_image',
       'profile_should_show_feature_image_section',
       'profile_feature_image_attributes_image_urls_default',
       'profile_feature_image_attributes_image_urls_baseball_card', 'slug',
       'source_url', 'staff_pick', 'static_usd_rate', 'urls_web_project',
       'urls_web_rewards', 'video', 'story_html', 'risks_html', 'rewards_html',
       'pledged', 'backers_count', 'usd_pledged', 'converted_pledged_amount',
       'profile

In [18]:
# check data types
print("\nData types in df:")
print(df.dtypes)


Data types in df:
blurb                                                            str
category_name                                                    str
category_parent_name                                             str
country                                                          str
created_at                                                       str
creator_has_admin_message_badge                              float64
creator_ppo_has_action                                       float64
creator_backing_action_count                                 float64
currency                                                         str
deadline                                                         str
disable_communication                                           bool
goal                                                         float64
id                                                             int64
is_disliked                                                     bool
is_liked       

In [ ]:
# define text columns and convert to string type
text_cols = [
    "blurb", "name", "story_html", "risks_html", "rewards_html", "slug"
]

df[text_cols] = df[text_cols].astype("string")

In [ ]:
# define categorical columns and convert to category type
categorical_cols = [
    "category_name", "category_parent_name", "country", "currency", "location_name", "location_state", "location_country", "location_type"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

In [21]:
# convert video column to binary
if 'video' in df.columns:
    df['video'] = df['video'].apply(lambda x: 1 if x == 'video exists' else 0)
    print("Converted video column to binary indicator.")

Converted video column to binary indicator.


In [ ]:
# convert profile_state to binary
df['profile_state'] = df['profile_state'].map({'active': True, 'inactive': False})

In [ ]:
# define boolean columns and convert to bool type
bool_cols = [
    'creator_has_admin_message_badge',
    'creator_ppo_has_action',
    'disable_communication',
    'is_disliked',
    'is_liked',
    'is_starrable',
    'prelaunch_activated',
    'profile_show_feature_image',
    'profile_should_show_feature_image_section',
    'video',
    'staff_pick',
    'spotlight',
    'is_english_project',
    'profile_state'
    ]

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)

In [24]:
# print head for bool cols
for col in bool_cols:
    if col in df.columns:
        print(f"\n{col} head:")
        print(df[col].head())


creator_has_admin_message_badge head:
0    False
1    False
2    False
3    False
4    False
Name: creator_has_admin_message_badge, dtype: bool

creator_ppo_has_action head:
0    False
1    False
2    False
3    False
4    False
Name: creator_ppo_has_action, dtype: bool

disable_communication head:
0    False
1    False
2    False
3    False
4    False
Name: disable_communication, dtype: bool

is_disliked head:
0    False
1    False
2    False
3    False
4    False
Name: is_disliked, dtype: bool

is_liked head:
0    False
1    False
2    False
3    False
4    False
Name: is_liked, dtype: bool

is_starrable head:
0    False
1    False
2    False
3    False
4    False
Name: is_starrable, dtype: bool

prelaunch_activated head:
0    False
1    False
2    False
3     True
4     True
Name: prelaunch_activated, dtype: bool

profile_show_feature_image head:
0    False
1    False
2    False
3    False
4    False
Name: profile_show_feature_image, dtype: bool

profile_should_show_feature_image_s

In [ ]:
# define URL columns and convert to string type
urls_cols = [
    "photo_1536x864",
    "profile_feature_image_attributes_image_urls_default",
    "profile_feature_image_attributes_image_urls_baseball_card",
    "source_url",
    "urls_web_project",
    "urls_web_rewards"
]
for col in urls_cols:
    if col in df.columns:
        df[col] = df[col].astype("string")

In [26]:
# datatype of goal and pledged columns
print(df[['goal', 'pledged']].dtypes)
df['goal'] = pd.to_numeric(df['goal'], errors='coerce')
df['pledged'] = pd.to_numeric(df['pledged'], errors='coerce')
print(df[['goal', 'pledged']].dtypes)

goal       float64
pledged    float64
dtype: object
goal       float64
pledged    float64
dtype: object


In [27]:
# show created_at and launched_at columns
print(df[['created_at', 'launched_at', 'deadline', "profile_state_changed_at", "state_changed_at"]].head())
# their data type
print(df[['created_at', 'launched_at', 'deadline', "profile_state_changed_at", "state_changed_at"]].dtypes)

            created_at          launched_at             deadline  \
0  2015-08-04 10:49:24  2015-08-11 10:12:28  2015-10-09 09:36:00   
1  2016-01-19 03:48:47  2016-02-26 12:38:27  2016-04-01 11:38:27   
2  2018-09-08 14:23:23  2018-09-11 08:38:01  2018-09-29 08:38:01   
3  2023-10-07 20:25:28  2024-07-02 19:11:53  2024-08-03 03:59:00   
4  2022-10-26 11:42:19  2022-11-07 15:04:01  2022-12-02 15:04:01   

  profile_state_changed_at     state_changed_at  
0      2015-08-04 10:49:24  2015-10-09 09:36:03  
1      2016-01-19 03:48:47  2016-04-01 11:38:28  
2      2018-09-08 14:23:23  2018-09-29 08:38:01  
3      2024-08-03 13:31:37  2024-08-03 03:59:00  
4      2022-10-26 11:42:19  2022-12-02 15:04:01  
created_at                  str
launched_at                 str
deadline                    str
profile_state_changed_at    str
state_changed_at            str
dtype: object


In [ ]:
# Convert to datetime
date_cols = ['created_at', 'launched_at', 'deadline', 'profile_state_changed_at', 'state_changed_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Check
print(df[date_cols].dtypes)

print(df[['created_at', 'launched_at', 'deadline', 'profile_state_changed_at', 'state_changed_at']].head())

created_at                  datetime64[us]
launched_at                 datetime64[us]
deadline                    datetime64[us]
profile_state_changed_at    datetime64[us]
state_changed_at            datetime64[us]
dtype: object
           created_at         launched_at            deadline  \
0 2015-08-04 10:49:24 2015-08-11 10:12:28 2015-10-09 09:36:00   
1 2016-01-19 03:48:47 2016-02-26 12:38:27 2016-04-01 11:38:27   
2 2018-09-08 14:23:23 2018-09-11 08:38:01 2018-09-29 08:38:01   
3 2023-10-07 20:25:28 2024-07-02 19:11:53 2024-08-03 03:59:00   
4 2022-10-26 11:42:19 2022-11-07 15:04:01 2022-12-02 15:04:01   

  profile_state_changed_at    state_changed_at  
0      2015-08-04 10:49:24 2015-10-09 09:36:03  
1      2016-01-19 03:48:47 2016-04-01 11:38:28  
2      2018-09-08 14:23:23 2018-09-29 08:38:01  
3      2024-08-03 13:31:37 2024-08-03 03:59:00  
4      2022-10-26 11:42:19 2022-12-02 15:04:01  


In [ ]:
# check data types again
print("Data types after conversion:")
print(df.dtypes)


Data types after conversion:
blurb                                                                string
category_name                                                      category
category_parent_name                                               category
country                                                            category
created_at                                                   datetime64[us]
creator_has_admin_message_badge                                        bool
creator_ppo_has_action                                                 bool
creator_backing_action_count                                        float64
currency                                                           category
deadline                                                     datetime64[us]
disable_communication                                                  bool
goal                                                                float64
id                                                        

In [30]:
# create a campaign duration column
df['campaign_duration_days'] = (df['deadline'] - df['launched_at']).dt.days
print(df[['created_at', 'launched_at', 'deadline', 'campaign_duration_days']].head())

           created_at         launched_at            deadline  \
0 2015-08-04 10:49:24 2015-08-11 10:12:28 2015-10-09 09:36:00   
1 2016-01-19 03:48:47 2016-02-26 12:38:27 2016-04-01 11:38:27   
2 2018-09-08 14:23:23 2018-09-11 08:38:01 2018-09-29 08:38:01   
3 2023-10-07 20:25:28 2024-07-02 19:11:53 2024-08-03 03:59:00   
4 2022-10-26 11:42:19 2022-11-07 15:04:01 2022-12-02 15:04:01   

   campaign_duration_days  
0                      58  
1                      34  
2                      18  
3                      31  
4                      25  


In [ ]:
# variable that calculated days between created_at and launched_at
df['days_between_created_launched'] = (pd.to_datetime(df['launched_at']) - pd.to_datetime(df['created_at'])).dt.days
print("days_between_created_launched column created.")
# check results
df['days_between_created_launched'].head()

days_between_created_launched column created.


0      6
1     38
2      2
3    268
4     12
Name: days_between_created_launched, dtype: int64

In [ ]:
# create variables that extract day of launched_at and hour of launch
df['launched_day'] = df['launched_at'].dt.day
df['launched_day_of_week'] = df['launched_at'].dt.dayofweek  # 0=Monday, 6=Sunday
df['launched_hour'] = df['launched_at'].dt.hour
df['launched_month'] = df['launched_at'].dt.month

print("launched_day column created.")
print("launched_day_of_week column created.")
print("launched_hour column created.")
print("launched_month column created.")

# check results
print(df[['launched_at', 'launched_day', 'launched_day_of_week', 'launched_hour', 'launched_month']].head(10))

# summary statistics
print("Summary stats:")
print("Day of month range:", df['launched_day'].min(), "-", df['launched_day'].max())
print("Hour range:", df['launched_hour'].min(), "-", df['launched_hour'].max())
print("Day of week distribution:", df['launched_day_of_week'].value_counts().sort_index())
print("Month distribution:", df['launched_month'].value_counts().sort_index())
print("Hour distribution:", df['launched_hour'].value_counts().sort_index())

launched_day column created.
launched_day_of_week column created.
launched_hour column created.
launched_month column created.
          launched_at  launched_day  launched_day_of_week  launched_hour  \
0 2015-08-11 10:12:28            11                     1             10   
1 2016-02-26 12:38:27            26                     4             12   
2 2018-09-11 08:38:01            11                     1              8   
3 2024-07-02 19:11:53             2                     1             19   
4 2022-11-07 15:04:01             7                     0             15   
5 2019-09-25 22:47:29            25                     2             22   
6 2025-03-04 13:55:15             4                     1             13   
7 2017-08-07 23:20:20             7                     0             23   
8 2024-10-08 14:05:01             8                     1             14   
9 2014-11-14 17:02:00            14                     4             17   

   launched_month  
0               

In [33]:
# create goal_converted_USD column
df['goal_converted_USD'] = df['goal'] * df['static_usd_rate']
print("goal_converted_USD column created.")

goal_converted_USD column created.


In [ ]:
# Quick diagnostics on goal_converted_USD
print("dtype:", df['goal_converted_USD'].dtype)
pd.options.display.float_format = '{:,.2f}'.format
print(df['goal_converted_USD'].describe())
print("min, 25%, 50%, 75%, max:", df['goal_converted_USD'].min(), df['goal_converted_USD'].quantile(0.25),
      df['goal_converted_USD'].median(), df['goal_converted_USD'].quantile(0.75), df['goal_converted_USD'].max())
print("Number of zeros:", (df['goal_converted_USD'] == 0).sum())
print("Number of negatives:", (df['goal_converted_USD'] < 0).sum())

dtype: float64
count        44,995.00
mean         44,107.16
std       1,226,630.01
min               0.65
25%           1,500.00
50%           5,000.00
75%          13,518.75
max     118,269,015.00
Name: goal_converted_USD, dtype: float64
min, 25%, 50%, 75%, max: 0.65342181 1500.0 5000.0 13518.754217704998 118269015.0
Number of zeros: 0
Number of negatives: 0


In [35]:
# convert goal_converted_USD to log scale
df['goal_converted_USD_log'] = df['goal_converted_USD'].apply(lambda x: np.log1p(x) if pd.notnull(x) and x > 0 else x)
print("Converted goal_converted_USD to log scale.")

Converted goal_converted_USD to log scale.


In [36]:
# add usd_pledge_per_backer column
df['usd_pledge_per_backer'] = df['usd_pledged'] / df['backers_count']
print(df[['usd_pledged', 'backers_count', 'usd_pledge_per_backer']].head())

   usd_pledged  backers_count  usd_pledge_per_backer
0         0.00              0                    NaN
1    52,375.00            224                 233.82
2       684.42             42                  16.30
3    10,069.00            111                  90.71
4    30,722.00            148                 207.58


In [ ]:
# missing values in usd_pledged, backers_count
print("Missing values before calculating usd_pledge_per_backer:")
print(df[['usd_pledged', 'backers_count']].isnull().sum())

# Count the number of rows where backers_count is zero
zero_backers_count = (df['backers_count'] == 0).sum()
print(f"Number of projects with 0 backers: {zero_backers_count}")

# Check missing values after calculating usd_pledge_per_backer
print("Missing values after calculating usd_pledge_per_backer:")
print(df[['usd_pledged', 'backers_count', 'usd_pledge_per_backer']].isnull().sum())

Missing values before calculating usd_pledge_per_backer:
usd_pledged      0
backers_count    0
dtype: int64

Number of projects with 0 backers: 2996

Missing values after calculating usd_pledge_per_backer:
usd_pledged                 0
backers_count               0
usd_pledge_per_backer    2996
dtype: int64


In [ ]:
# Fill missing values in usd_pledge_per_backer with 0
df['usd_pledge_per_backer'] = df['usd_pledge_per_backer'].fillna(0)

# Verify that the missing values have been filled
print("Missing values in usd_pledge_per_backer after filling:")
print(df['usd_pledge_per_backer'].isnull().sum())

Missing values in usd_pledge_per_backer after filling:
0


In [39]:
# number of rows with state failed and successful
num_failed = df[df['state'] == 'failed'].shape[0]
print(f"Number of failed projects: {num_failed}")
num_successful = df[df['state'] == 'successful'].shape[0]
print(f"Number of successful projects: {num_successful}")

Number of failed projects: 22454
Number of successful projects: 22541


In [ ]:
# create binary target variable for success
df['success'] = (df['state'] == 'successful').astype(int)
print(df[['state', 'success']].head(10))

        state  success
0      failed        0
1  successful        1
2  successful        1
3  successful        1
4  successful        1
5      failed        0
6  successful        1
7      failed        0
8  successful        1
9  successful        1


Text Cleaning

In [ ]:
# Function to count reward tiers from rewards_html based on <article> elements with id starting 'reward-'
def count_reward_tiers(rewards_html):
    if not rewards_html or not isinstance(rewards_html, str):
        return 0

    soup = BeautifulSoup(rewards_html, "html.parser")

    reward_articles = soup.find_all(
        "article",
        id=lambda x: isinstance(x, str) and x.startswith("reward-")
    )

    return len(reward_articles)

# call function
df['num_reward_tiers'] = df['rewards_html'].apply(count_reward_tiers)

In [42]:
print(df[["num_reward_tiers"]].head())

   num_reward_tiers
0                 1
1                16
2                 3
3                 4
4                 4


In [ ]:
# summary statistics for num_reward_tiers
df["num_reward_tiers"].describe()

count   44,995.00
mean         7.21
std          5.62
min          0.00
25%          4.00
50%          6.00
75%          9.00
max        100.00
Name: num_reward_tiers, dtype: float64

In [ ]:
PRICE_REGEX = re.compile(r"([\d,.]+)")

# Function to extract lowest and highest pledge levels from rewards_html and convert to USD
def extract_pledge_levels_usd(rewards_html, static_usd_rate):
    if not rewards_html or not isinstance(rewards_html, str):
        return 0.0, 0.0

    if not static_usd_rate or static_usd_rate <= 0:
        return 0.0, 0.0

    soup = BeautifulSoup(rewards_html, "html.parser")
    pledge_amounts = []

    reward_articles = soup.find_all(
        "article",
        id=lambda x: isinstance(x, str) and x.startswith("reward-")
    )

    for article in reward_articles:
        price_tag = article.find("p", string=PRICE_REGEX)
        if not price_tag:
            continue

        match = PRICE_REGEX.search(price_tag.get_text(strip=True))
        if not match:
            continue

        amount = match.group(1).replace(",", "")

        try:
            amount_local = float(amount)
            amount_usd = amount_local * static_usd_rate
            pledge_amounts.append(amount_usd)
        except ValueError:
            continue

    if not pledge_amounts:
        return 0.0, 0.0

    return min(pledge_amounts), max(pledge_amounts)

# call function
df[["lowest_pledge_level_usd", "highest_pledge_level_usd"]] = (
    df.apply(
        lambda row: pd.Series(
            extract_pledge_levels_usd(
                row["rewards_html"],
                row["static_usd_rate"]
            )
        ),
        axis=1
    )
)
print(df[["lowest_pledge_level_usd", "highest_pledge_level_usd"]].head())

   lowest_pledge_level_usd  highest_pledge_level_usd
0                 1,548.64                  1,548.64
1                    10.00                  5,000.00
2                    11.02                     55.12
3                    25.00                    125.00
4                   109.00                    239.00


In [ ]:
# summary statistics for lowest and highest pledge levels in USD
df[["lowest_pledge_level_usd", "highest_pledge_level_usd"]].describe()

,lowest_pledge_level_usd,highest_pledge_level_usd
count,"44,995.00","44,995.00"
mean,44.04,"1,357.58"
std,342.22,"2,454.47"
min,0.00,0.00
25%,5.00,95.27
50%,10.00,300.00
75%,20.00,"1,000.00"
max,"10,703.13","35,000.00"


In [ ]:
# check for reward images in rewards_html
def has_reward_main_images(rewards_html, min_width=50, min_height=50):
    if not rewards_html or not isinstance(rewards_html, str):
        return False

    soup = BeautifulSoup(rewards_html, "html.parser")
    reward_articles = soup.find_all(
        "article",
        id=lambda x: isinstance(x, str) and x.startswith("reward-")
    )

    for article in reward_articles:
        for img in article.find_all("img"):
            width = img.get("width")
            height = img.get("height")
            if width and height:
                if int(width) >= min_width and int(height) >= min_height:
                    return True
            else:
                # check if img is inside a div with 'aspect-ratio' class
                if img.find_parent("div", class_=lambda x: x and "aspect-ratio" in x):
                    return True

    return False

df["has_reward_images"] = df["rewards_html"].apply(has_reward_main_images)

print(df[["has_reward_images"]].head(10))

   has_reward_images
0              False
1              False
2              False
3               True
4              False
5              False
6               True
7              False
8               True
9              False


In [ ]:
# Function to calculate average number of words per reward tier
def average_words_per_reward(html):
    if pd.isna(html) or html.strip() == "":
        return 0

    soup = BeautifulSoup(html, "html.parser")

    reward_descriptions = []
    for article in soup.find_all("article", id=lambda x: x and x.startswith("reward-")):
        # Try to find text within common tags used for descriptions
        description_text = ""
        # Prioritize divs with specific classes if they exist
        for div in article.select("div.type-14.lh20px.mb0.support-700.text-prewrap"):
            description_text += div.get_text(strip=True) + " "
        # If not found or empty, look for any paragraph or span text within the article
        if not description_text.strip():
            for p in article.find_all("p"): 
                description_text += p.get_text(strip=True) + " "
            for span in article.find_all("span"):
                description_text += span.get_text(strip=True) + " "
        
        if description_text.strip():
            reward_descriptions.append(description_text.strip())

    if not reward_descriptions:
        return 0

    # Count words in each reward
    word_counts = [len(desc.split()) for desc in reward_descriptions]

    # Return the average
    return np.mean(word_counts)

# Apply
df["average_number_of_words_per_reward"] = df["rewards_html"].apply(average_words_per_reward)
df[["average_number_of_words_per_reward"]].head()

,average_number_of_words_per_reward
0,9.00
1,13.62
2,35.00
3,51.50
4,28.00


In [ ]:
# Show full text in the rewards_html column
pd.set_option('display.max_colwidth', None)
# Print the entire column
df['rewards_html'].head()

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
# Show full text in the rewards_html column
pd.set_option('display.max_colwidth', None)
# Print the entire column
df['rewards_html_text'].head()

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
# Function to check if any reward tier has restricted shipping (not "Anywhere in the world")
def has_restricted_shipping(html):
    if pd.isna(html) or html.strip() == "":
        return np.nan

    soup = BeautifulSoup(html, "html.parser")

    ships_to_divs = soup.select("div.flex1 div.type-14.lh20px.mb0.support-700")

    for div in ships_to_divs:
        text = div.get_text(strip=True)
        if text.lower() != "anywhere in the world":
            return 1  # At least one reward has restricted shipping

    return 0  # All rewards ship anywhere

# Apply function
df["has_restricted_shipping"] = df["rewards_html"].apply(has_restricted_shipping)
df[["has_restricted_shipping"]].head()

,has_restricted_shipping
0,0
1,1
2,0
3,1
4,0


In [ ]:
# Function to extract and clean text from rewards_html, keeping only reward descriptions
def extract_rewards_only(html):
    if not html or not isinstance(html, str):
        return np.nan

    soup = BeautifulSoup(html, "html.parser")

    # Find all <article> tags that correspond to reward tiers
    reward_articles = soup.find_all("article", id=lambda x: x and x.startswith("reward-"))

    # Extract text from each reward and join
    rewards_text = " ".join(article.get_text(separator=" ", strip=True) for article in reward_articles)

    # Remove \n and \r, then collapse multiple spaces into one
    rewards_text = rewards_text.replace('\n', ' ').replace('\r', ' ')
    rewards_text = re.sub(r'\s+', ' ', rewards_text).strip()

    if rewards_text == "":
        return np.nan

    return rewards_text
# Apply
df["rewards_html_text"] = df["rewards_html"].apply(extract_rewards_only)

In [59]:
pd.set_option('display.max_colwidth', None)
df[["rewards_html_text","urls_web_project"]].head(10)

,rewards_html_text,urls_web_project
0,"£1,000 Money back plus a copy of the limited edition Pledge £1,000",https://www.kickstarter.com/projects/753774991/the-songs-of-adelaide-and-abullah
1,"$10 Free cup of coffee at Monarch $25 Printed Monarch mug $50 Chocolate + coffee pairing for you and your BAE or BFF// Try 3 different kinds of coffee drinks paired with chocolates. $75 Handmade ceramic mug dipped in indigo. Each unique mug is created in the South by our ceramicist. $80 Two tickets to our opening party. Coffee drinks, treats, entertainment on us. $100 Milk + sugar ceramic set with wooden spoon $150 1 free coffee a day for a month at Monarch + a printed Monarch coffee mug. Mug will be provided on your first day. $200 Latte art course for you + 3 friends + 4 printed Monarch mugs. Mugs are provided at latte art course. $250 Your signature + favorite lyric or quote engraved on our wall. ""These are the people who believed in Monarch."" $300 Coffee club. One seasonal, single origin coffee bag from our roaster shipped to your house monthly for a year. $500 Speciality drink named after you + coffee club for one year. $1,000 Private opening party for you and 10 friends. $1,250 Free coffee on gamedays (home or away) for a lifetime. $1,500 Coffee club for 5 years. One single origin, seasonal coffee bag shipped to your house every month for 5 years. $2,000 Lifetime supply of coffee at Monarch. // Includes pour overs and drip coffee. Excludes espresso-based drinks. $5,000 All of the above.",https://www.kickstarter.com/projects/362504450/monarch-espresso-bar
2,"Ett ex av Fozzie nr. 12 SEK 100 Stöttar du projektet med minst 100 kronor skickar jag dig ett ex av Fozzie nr. 12 så fort det är färdigt! Tack!! Pledge SEK 100 Fozzie 12 med specialinlaga SEK 200 Som tack för att du bidrar med minst 200 kronor får du ett ex av Fozzie nr. 12 med en exklusiv specialinlaga med en av mina bästa intervjuer från de tidigare numren. Pledge SEK 200 Sponsor + tidningen med specialinlaga SEK 500 Stöttar du Fozzie med minst 500 kronor blir du en av de största sponsorerna, och får självklart ett hedersomnämnande i tidningen. Tusen tack! Du får ett ex av Fozzie nr. 12, med en exklusiv specialinlaga med en av mina bästa intervjuer från de tidigare numren – och ditt namn i tidningen. Pledge SEK 500",https://www.kickstarter.com/projects/1040055915/fozzie-fanzine-nummer-12-svensk-indiepop
3,"Starter Pack $25 The Ferret Flush® Original pack was designed with adolescent and young adult-aged players in mind (13+), but we’ve had players of all ages enjoy a few rounds! Ferret Flush® was created as a competitive card game to bring people together in play and laughter, but also as an educational tool to start meaningful conversations around our thoughts, feelings and actions. Play with your friends, your children, your friend’s children, your clients, your colleagues, your hairdresser. Teachers, coaches, mentors and well-being workers also enjoy using this pack to facilitate discussions. Fan Pack $50 With the Fan Pack you will receive two Ferret Flush card games of your choosing; Original, Junior, and Couples Packs. You will also receive a sticker! Friend Pack $75 The Ferret Flush® Friend Pack is a $75 backing that rewards you with all three unique card games; Original, Junior, and Couples, plus a sticker! Adventure Bag $125 With the Ferret Flush® Adventure Bag, you receive a limited-edition enamel Ferret Pin! And you'll get all three of the Ferret Flush® Packs; Original, Junior, and Couples pack, plus two Ferret Out™ Journals, two pencils, two wristbands, and five stickers. All the goodies come with a drawstring bag to play and grow in community! (The featured image does not display the correct quantities of each item.)",https://www.kickstarter.com/projects/bviolagonzalez/ferret-flush
4,Hakii Mix - Early Bird x 1 $109 Get your hands on the Hakii Mix with 38% off. The retail price is $176 Including: -Hakii Mix headphone -Detachable elastic band(S/M/L) -USB-C charging c

In [ ]:
# Function to clean rewards_html_text by removing metadata, all placeholders and keeping only reward descriptions
def clean_rewards_plain_text(text):
    if pd.isna(text) or not str(text).strip():
        return np.nan

    text = str(text)

    # Patterns to remove
    patterns = [
        r'Backers\s+\d+',  # Backers count
        r'Estimated delivery.*?(?=Backers|Ships to|Limited quantity|Details|Pledge|$)',
        r'Ships to.*?(?=Estimated|Backers|Limited quantity|Details|Pledge|$)',
        r'Limited quantity.*?(?=Backers|Estimated|Details|Pledge|$)',
        r'\d+\s+items included.*?(?=\+|$)',
        r'\+\d+\s+More',
        r'\+\s*optional add-ons',
        r'Details',
        r'Pledge US\$\s?[\d,]+',
        r'Funding period.*',
        r'All gone'
    ]

    # Apply all patterns
    for pattern in patterns:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    if text == "":
        return np.nan

    return text
# Apply to rewards_html_text
df["rewards_html_text"] = df["rewards_html_text"].apply(clean_rewards_plain_text)

In [52]:
df[["rewards_html_text", "urls_web_project"]].head(10)

,rewards_html_text,urls_web_project
0,"£1,000 Money back plus a copy of the limited e...",https://www.kickstarter.com/projects/753774991...
1,$10 Free cup of coffee at Monarch $25 Printed ...,https://www.kickstarter.com/projects/362504450...
2,Ett ex av Fozzie nr. 12 SEK 100 Stöttar du pro...,https://www.kickstarter.com/projects/104005591...
3,Starter Pack $25 The Ferret Flush® Original pa...,https://www.kickstarter.com/projects/bviolagon...
4,Hakii Mix - Early Bird x 1 $109 Get your hands...,https://www.kickstarter.com/projects/hakii/hak...
5,donation $5 Donate and your name will appear o...,https://www.kickstarter.com/projects/154216094...
6,Digital #4 $8 24 pages of western fantasy adve...,https://www.kickstarter.com/projects/giraffe-d...
7,Free shirt (pay postage) AU$ 100 Any donations...,https://www.kickstarter.com/projects/194262678...
8,Haunted Harvest .STL Files - Early Bird £5 Wit...,https://www.kickstarter.com/projects/js3ddesig...
9,CA$ 30 EARLY BIRD - Streaming access to the en...,https://www.kickstarter.com/projects/oasselin/...


In [54]:
# length of rewards_html_text
df['rewards_char_length'] = df['rewards_html_text'].apply(lambda x: len(x) if isinstance(x, str) else 0)

In [56]:
print(df["risks_html"].head())

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      <div class="mobile-show py3"></div><h2 class="normal mb4">Risks and challenges</h2><p class="js-risks-text text-preline">No risk. Money will be re-paid as per schedule in the event of no sale of books. Poems are already written</p><a target="_blank" class="ksr-green-700 bold type-16" href="/help/hc/sections/115001107133">Learn about accountability on Kickstarter</a>
1                                                                                                                                               

In [ ]:
# Function to clean risks_html by removing the "Risks and challenges" header and the "Learn about accountability on Kickstarter" link
def clean_risks_html(html):
    if pd.isna(html) or not html.strip():
        return html

    soup = BeautifulSoup(html, "html.parser")

    # Remove the "Risks and challenges" header
    for h2 in soup.find_all("h2"):
        if h2.get_text(strip=True).lower() == "risks and challenges":
            h2.decompose()

    # Remove the "Learn about accountability on Kickstarter" link
    for a in soup.find_all("a"):
        if "learn about accountability on kickstarter" in a.get_text(strip=True).lower():
            a.decompose()

    return str(soup)
df["risks_html"] = df["risks_html"].apply(clean_risks_html)

/tmp/ipykernel_834/1767599699.py:5: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(html, "html.parser")


In [ ]:
# Function to extract and clean text from risks_html, keeping only the risk descriptions
def extract_risks_only(html):
    if not html or not isinstance(html, str):
        return np.nan

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r'\s+', ' ', text).strip()

    if text == "":
        return np.nan

    return text
df["risks_html_text"] = df["risks_html"].apply(extract_risks_only)

/tmp/ipykernel_834/3968637462.py:5: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(html, "html.parser")


In [63]:
df[["risks_html", "risks_html_text"]].head()

,risks_html,risks_html_text
0,"<div class=""mobile-show py3""></div><p class=""js-risks-text text-preline"">No risk. Money will be re-paid as per schedule in the event of no sale of books. Poems are already written</p>",No risk. Money will be re-paid as per schedule in the event of no sale of books. Poems are already written
1,"<div class=""mobile-show py3""></div><p class=""js-risks-text text-preline"">The challenges that come with our project are unforeseen updates needed on the building. If we are fully funded, we will move forward on our location even if it means building out part of the space and leaving certain rooms untouched until we're able to fund those renovations.\r\n\r\nThankfully, we will still be able to operate in our space without renovating the entire building at once.</p>","The challenges that come with our project are unforeseen updates needed on the building. If we are fully funded, we will move forward on our location even if it means building out part of the space and leaving certain rooms untouched until we're able to fund those renovations. Thankfully, we will still be able to operate in our space without renovating the entire building at once."
2,"<div class=""mobile-show py3""></div><p class=""js-risks-text text-preline"">Jag har bara några få intervjuer kvar att göra, och de är bokade inom de närmsta veckorna, så tidningen kommer bli färdig under september.</p>","Jag har bara några få intervjuer kvar att göra, och de är bokade inom de närmsta veckorna, så tidningen kommer bli färdig under september."
3,"<div class=""mobile-show py3""></div><p class=""js-risks-text text-preline"">CHALLENGES\n1. Funding to print each edition of the card game.\n2. Printing in the US is more expensive than printing oversees, almost double the cost.\n3. Breaking into stores like Target, Barnes &amp; Noble, Walmart, etc.\n\nRISKS\nPrinting timelines range based on the amount of units printed from 3-5 weeks (2,000 units) to 3 months (10,000 units)</p>","CHALLENGES 1. Funding to print each edition of the card game. 2. Printing in the US is more expensive than printing oversees, almost double the cost. 3. Breaking into stores like Target, Barnes & Noble, Walmart, etc. RISKS Printing timelines range based on the amount of units printed from 3-5 weeks (2,000 units) to 3 months (10,000 units)"
4,"<div class=""mobile-show py3""></div><p class=""js-risks-text text-preline"">Timing is always a challenge with the production of a new product.\nHowever, we are working with top-notch suppliers and we have padded our timeline to make sure we can hit the time frames we have laid out.\n\nAlso, our QA team is in place to personally check every shipment that will go out to make sure all backers receive the product without any defaults.\n\nThe challenge, therefore, will not be the production side, but mostly the fulfilment side. We now have a good margin in our delivery times, and hope to deliver all the backpacks even before the deadline mentioned on this page.</p>","Timing is always a challenge with the production of a new product. However, we are working with top-notch suppliers and we have padded our timeline to make sure we can hit the time frames we have laid out. Also, our QA team is in place to personally check every shipment that will go out to make sure all backers receive the product without any defaults. The challenge, therefore, will not be the production side, but mostly the fulfilment side. We now have a good margin in our delivery times, and hope to deliver all the backpacks even before the deadline mentioned on this page."


In [65]:
# length of risks_html_text
df['risks_char_length'] = df['risks_html_text'].apply(lambda x: len(x) if isinstance(x, str) else 0)

In [66]:
df['risks_char_length'].head()

0    106
1    383
2    138
3    340
4    581
Name: risks_char_length, dtype: int64

In [ ]:
STATIC_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
ANIMATED_EXTS = (".gif",)

# Function to normalize URLs by removing query parameters and fragments, and converting to lowercase
def normalize_url(url: str) -> str:
    if not url:
        return ""
    parsed = urlparse(url)
    return urlunparse(parsed._replace(query="", fragment="")).lower()

# Function to compute total media asset counts, static image counts, and animated content counts from story_html
def compute_media_counts(story_html: str):
    if not story_html or not isinstance(story_html, str):
        return 0, 0, 0

    soup = BeautifulSoup(story_html, "html.parser")
    static_images = set()
    animated_content = set()

    # Process all images
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or img.get("data-lazy") or img.get("data-original")
        if not src:
            continue
        base = normalize_url(src)

        # Skip images that are part of video containers
        if img.find_parent("div", class_="video-player") or img.find_parent("video"):
            continue

        # Check for animated GIFs
        if base.endswith(ANIMATED_EXTS):
            animated_content.add(base)
        # Otherwise consider static images
        elif base.endswith(STATIC_EXTS):
            static_images.add(base)

    # Process videos
    for video_div in soup.find_all("div", class_="video-player"):
        # Extract the main video URL from the data attribute
        src = video_div.get("data-video-url")
        if src:
            animated_content.add(normalize_url(src))

    # total counts
    total_assets = len(static_images) + len(animated_content)
    return total_assets, len(static_images), len(animated_content)

df[["total_media_assets", "num_static_images", "num_animated_content"]] = df["story_html"].apply(
    lambda html: pd.Series(compute_media_counts(html)))

In [74]:
df[["total_media_assets", "num_static_images", "num_animated_content"]].head()

,total_media_assets,num_static_images,num_animated_content
0,0,0,0
1,15,14,1
2,4,4,0
3,35,35,0
4,19,12,7


In [ ]:
# Function to extract and clean text from story_html, keeping only the story description and removing all media-related content
def extract_story_only(html):
    if pd.isna(html) or not isinstance(html, str):
        return np.nan

    soup = BeautifulSoup(html, "html.parser")

    # Remove image-related tags completely
    for tag in soup.find_all(["img", "figure", "video"]):
        tag.decompose()

    # Extract all remaining text
    text = soup.get_text(separator=" ", strip=True)

    text = re.sub(r'\s+', ' ', text).strip()

    if text == "":
        return np.nan

    return text
# apply
df["story_html_text"] = df["story_html"].apply(extract_story_only)

In [76]:
df[["story_html_text"]].head()

story_html_text
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [77]:
# calculate the length of story_html_text
df['story_char_length'] = df['story_html_text'].apply(
    lambda x: len(x) if isinstance(x, str) else np.nan
)
df['story_char_length'].head()

0       1.00
1   4,575.00
2   1,950.00
3   4,960.00
4     996.00
Name: story_char_length, dtype: float64

In [ ]:
# length of blurb
df['blurb_char_length'] = df['blurb'].apply(
    lambda x: len(x) if isinstance(x, str) else np.nan
)
print(df['blurb_char_length'].head())

0    133
1     77
2    135
3    121
4    118
Name: blurb_char_length, dtype: int64


In [ ]:
# calculate the ratio of text length to total media assets
df['ratio_text_to_total_assets'] = (
    df['story_char_length']
    .div(df['total_media_assets'].replace(0, np.nan))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [81]:
# print all columns
df.columns.to_list()

['blurb',
 'category_name',
 'category_parent_name',
 'country',
 'created_at',
 'creator_has_admin_message_badge',
 'creator_ppo_has_action',
 'creator_backing_action_count',
 'currency',
 'deadline',
 'disable_communication',
 'goal',
 'id',
 'is_disliked',
 'is_liked',
 'is_starrable',
 'launched_at',
 'location_name',
 'location_country',
 'location_state',
 'location_type',
 'name',
 'photo_1536x864',
 'prelaunch_activated',
 'profile_id',
 'profile_project_id',
 'profile_show_feature_image',
 'profile_should_show_feature_image_section',
 'profile_feature_image_attributes_image_urls_default',
 'profile_feature_image_attributes_image_urls_baseball_card',
 'slug',
 'source_url',
 'staff_pick',
 'static_usd_rate',
 'urls_web_project',
 'urls_web_rewards',
 'video',
 'story_html',
 'risks_html',
 'rewards_html',
 'pledged',
 'backers_count',
 'usd_pledged',
 'converted_pledged_amount',
 'profile_state',
 'profile_state_changed_at',
 'state_changed_at',
 'spotlight',
 'percent_funded',

In [ ]:
# check missing values
pd.set_option('display.max_rows', None)
print(df.isnull().sum())

blurb                                                          0
category_name                                                  0
category_parent_name                                           0
country                                                        0
created_at                                                     0
creator_has_admin_message_badge                                0
creator_ppo_has_action                                         0
creator_backing_action_count                                   0
currency                                                       0
deadline                                                       0
disable_communication                                          0
goal                                                           0
id                                                             0
is_disliked                                                    0
is_liked                                                       0
is_starrable             

In [ ]:
# remove project with images only in story and no text
df = df[~((df['story_char_length'] == 0) & (df['total_media_assets'] > 0))]

In [83]:
# delete row with missing value in story_html (they only have pictures, no text)
df = df.dropna(subset=['story_html_text']).reset_index(drop=True)
print("Final shape after dropping rows with missing story text:", df.shape)

Final shape after dropping rows with missing story text: (44441, 77)


In [86]:
# delete row with missing value in rewards_html_text
df = df.dropna(subset=['rewards_html_text']).reset_index(drop=True)
print("Final shape after dropping rows with missing rewards text:", df.shape)

Final shape after dropping rows with missing rewards text: (44345, 77)


Remove non-english projects

In [ ]:
NON_LATIN_REGEX = re.compile(r'[^\u0000-\u024F\u1E00-\u1EFF\s\d' + re.escape(string.punctuation) + ']')

# function to calculate the ratio of Latin characters to total characters in a text
def latin_ratio(text):
    if pd.isna(text) or not text.strip():
        return 1.0
    non_latin = len(NON_LATIN_REGEX.findall(text))
    return 1 - (non_latin / max(len(text), 1))

In [ ]:
def get_lang_detector(nlp, name):
    return LanguageDetector()

# Start with a blank English model
nlp = spacy.blank("en")

if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

if not Language.has_factory("language_detector"):
    Language.factory("language_detector", func=get_lang_detector)

if "language_detector" not in nlp.pipe_names:
    nlp.add_pipe("language_detector", last=True)

In [ ]:
# Function to check if text is English using spaCy's language detection, with a minimum probability threshold
def is_english_spacy_safe(text, min_prob=0.70):
    if pd.isna(text) or not text.strip():
        return True

    doc = nlp(text[:1000]) # Limit text length
    lang = doc._.language
    return lang["language"] == "en" and lang["score"] >= min_prob

In [ ]:
# Function to determine if a project is English based on blurb and story text, using Latin character ratio and spaCy language detection
def project_is_english(row, blurb_min_prob=0.90, story_min_prob=0.80, min_latin_ratio=0.90):
    blurb = str(row.get('blurb', ''))
    story = str(row.get('story_html_text', ''))

    # Check blurb
    if blurb.strip():
        if not (latin_ratio(blurb) >= min_latin_ratio and is_english_spacy_safe(blurb, blurb_min_prob)):
            return False

    # Check story
    if story.strip():
        story_sample = story[:2000] # limit story length
        if not (latin_ratio(story_sample) >= min_latin_ratio and is_english_spacy_safe(story_sample, story_min_prob)):
            return False

    return True

In [ ]:
df['is_english_project'] = df.apply(project_is_english, axis=1)

print("Projects before:", len(df))

# write non-english to new df
non_english_df = df[~df['is_english_project']]

# number of non-english projects
non_english_count = len(non_english_df)
print(f"Number of non-English projects: {non_english_count}")

Projects before: 44345
Number of non-English projects: 3874


In [98]:
# number of non-english projects
non_english_count = len(non_english_df)
print(f"Number of non-English projects: {non_english_count}")

Number of non-English projects: 3874


In [ ]:
# print non-english name, story_html_text and url
pd.set_option('display.max_colwidth', None)
print("Name:", non_english_df['name'].head(50))
print("Story:", non_english_df['story_html'].head(50))
print("URL:", non_english_df['urls_web_project'].head(50))

Name: 0                                  The Songs of Adelaide & Abullah
2                       Fozzie fanzine nummer 12 - svensk indiepop
25                                                    HIIT Bottle™
26                      Nuevo EP de corridos: La ciudad es nuestra
29                                                            CERO
47                                                  Blæs På Bølgen
48                                                Alpha Omega E.P.
50               VAGABONDES un livre de photographies d'Eric Facon
60                               Musical stage (escenario musical)
64                                                            Roxy
66                                                 Operación Douve
71                                                        COVENANT
82       APP - CUSTOM SHOES/ Calzado Personalizado - MADAME MARIAN
93                                       Handcrafted Spanking Tool
95                            Conference Paradigm Shift 

In [100]:
# check missing values
print(df.isnull().sum())

blurb                                                          0
category_name                                                  0
category_parent_name                                           0
country                                                        0
created_at                                                     0
creator_has_admin_message_badge                                0
creator_ppo_has_action                                         0
creator_backing_action_count                                   0
currency                                                       0
deadline                                                       0
disable_communication                                          0
goal                                                           0
id                                                             0
is_disliked                                                    0
is_liked                                                       0
is_starrable             

In [101]:
non_english_df.to_csv('non_english_projects.csv', index=False)

In [113]:
# save all this to new csv
df.to_csv('clean_df_final_preprocessed.csv', index=False)

---

In [114]:
# load csv
clean_df = pd.read_csv('clean_df_final_preprocessed.csv')
print("Final preprocessed DataFrame shape:", clean_df.shape)

Final preprocessed DataFrame shape: (44345, 78)


In [ ]:
# check missing
pd.set_option('display.max_rows', None)
print(clean_df.isnull().sum())

blurb                                                          0
category_name                                                  0
category_parent_name                                           0
country                                                        0
created_at                                                     0
creator_has_admin_message_badge                                0
creator_ppo_has_action                                         0
creator_backing_action_count                                   0
currency                                                       0
deadline                                                       0
disable_communication                                          0
goal                                                           0
id                                                             0
is_disliked                                                    0
is_liked                                                       0
is_starrable             

In [116]:
non_english_df = pd.read_csv('non_english_projects.csv')
print("Non-English projects DataFrame shape:", non_english_df.shape)

Non-English projects DataFrame shape: (3874, 78)


Remove non-english projects

In [ ]:
# drop non-english projects from clean_df
clean_df = clean_df[clean_df['is_english_project']].reset_index(drop=True)
print("Projects after:", len(clean_df))

Projects after: 40471


In [ ]:
# check missing values again
print(clean_df.isnull().sum())

blurb                                                          0
category_name                                                  0
category_parent_name                                           0
country                                                        0
created_at                                                     0
creator_has_admin_message_badge                                0
creator_ppo_has_action                                         0
creator_backing_action_count                                   0
currency                                                       0
deadline                                                       0
disable_communication                                          0
goal                                                           0
id                                                             0
is_disliked                                                    0
is_liked                                                       0
is_starrable             

Outlier Analysis

In [ ]:
# summary statistics for goal_converted_USD
clean_df['goal_converted_USD'].describe(percentiles=[0.50, 0.90, 0.95, 0.99, 0.995, 0.999])

count        40,471.00
mean         39,064.94
std         999,282.59
min               0.65
50%           5,000.00
90%          35,000.00
95%          65,000.00
99%         300,000.00
99.5%       667,863.27
99.9%     3,953,085.58
max     100,000,000.00
Name: goal_converted_USD, dtype: float64

In [ ]:
# summary statistics for goal_converted_USD 
clean_df['goal_converted_USD'].describe()

count        40,471.00
mean         39,064.94
std         999,282.59
min               0.65
25%           1,500.00
50%           5,000.00
75%          13,859.62
max     100,000,000.00
Name: goal_converted_USD, dtype: float64

In [126]:
# number of projects below and above the 1th and 99th percentile of goal_converted_USD
percentile_1 = clean_df['goal_converted_USD'].quantile(0.01)
percentile_99 = clean_df['goal_converted_USD'].quantile(0.99)
print(f"1th percentile of goal_converted_USD: {percentile_1}")
print(f"99th percentile of goal_converted_USD: {percentile_99}")
num_below_1th = (clean_df['goal_converted_USD'] < percentile_1).sum()
num_above_99th = (clean_df['goal_converted_USD'] > percentile_99).sum()
print(f"Number of projects below 1th percentile: {num_below_1th}")
print(f"Number of projects above 99th percentile: {num_above_99th}")

1th percentile of goal_converted_USD: 59.6588047
99th percentile of goal_converted_USD: 300000.0
Number of projects below 1th percentile: 405
Number of projects above 99th percentile: 392


In [ ]:
# summary statistics for usd_pledge_per_backer
print(clean_df['usd_pledge_per_backer'].describe())

count   40,471.00
mean        89.72
std        226.73
min          0.00
25%         20.00
50%         49.34
75%         92.67
max      9,370.74
Name: usd_pledge_per_backer, dtype: float64


In [ ]:
# outlier detection based on IQR method for goal_converted_USD
Q1 = clean_df['goal_converted_USD'].quantile(0.25)
Q3 = clean_df['goal_converted_USD'].quantile(0.75)
IQR = Q3 - Q1

# number of outliers based on IQR method
outliers = clean_df[(clean_df['goal_converted_USD'] < (Q1 - 1.5 * IQR)) | (clean_df['goal_converted_USD'] > (Q3 + 1.5 * IQR))]
print("Number of outliers in 'goal_converted_USD' column based on IQR method:", outliers.shape[0])

Number of outliers in 'goal' column based on IQR method: 4388


In [129]:
# number of projects with goal > 1,000,000 in goal_converted_USD
high_goal_projects = clean_df[clean_df['goal_converted_USD'] > 1000000]
print("Number of projects with goal > 1,000,000 USD:", high_goal_projects.shape[0])

Number of projects with goal > 1,000,000 USD: 122


In [ ]:
# number of projects with a funding goal > $1,000,000, which were not at least 1% funded
print("Number of projects with a funding goal > $1,000,000, which were not at least 1% funded:", len(clean_df[(clean_df['goal_converted_USD'] > 1000000) & (clean_df['percent_funded'] < 1)]))

Number of projects with a funding goal > $1,000,000, which were not at least 1% funded: 120


In [ ]:
# projects with a funding goal > $1,000,000, which were not at least 1% funded and their state
print(clean_df[(clean_df['goal_converted_USD'] > 1000000) & (clean_df['percent_funded'] < 1)][['name', 'goal_converted_USD', 'percent_funded', 'state', 'urls_web_project']])

                                                               name  \
424                                      Maori MBT Awareness Centre   
1086                                          Trew Watch | Make 100   
1346                        INDIRA – THE FIRST INDIAN GRAND MUSICAL   
1748                                             Groove City Dallas   
2840           Enjoy The Old Children's Home: Historic Preservation   
2975                                                    Save Nature   
3136             Make Linkin Park Great Again - Hybrid Theory Style   
3423                                 COSY: The future of daily life   
3756                                                           Duel   
4057                       Sojo & Mizzim Intimate SoulMate Sessions   
5849                                 Intelligent Design: The Series   
6241                                                           0XYZ   
7006                                         Mete Mano Producciones   
7012  

In [132]:
# check number of projects with 'goal_converted_USD' >= 1,000,000, 'story_char_length' < 1000, 'backers_count' < 10
extreme_projects = clean_df[(clean_df['goal_converted_USD'] >= 1000000) & (clean_df['story_char_length'] < 1000) & (clean_df['backers_count'] < 10)]
print("Number of projects with 'goal_converted_USD' >= 1,000,000, 'story_char_length' < 1000, 'backers_count' < 10:", extreme_projects.shape[0])

Number of projects with 'goal_converted_USD' >= 1,000,000, 'story_char_length' < 1000, 'backers_count' < 10: 41


In [133]:
# print few examples
pd.set_option('display.max_colwidth', None)
print(extreme_projects[['name', 'goal_converted_USD', 'backers_count', 'usd_pledged','urls_web_project']])

                                                               name  \
324                                         Xybears The Last Heroes   
2975                                                    Save Nature   
3136             Make Linkin Park Great Again - Hybrid Theory Style   
8104                      Bertone cars collection, Museum in motion   
8205                                                     Games Over   
8583                               From Dawson to Jackson Vineyards   
9793                                     Help Save Parkway Cinemas!   
10357                                             Puppy Play Ground   
11835                           "Starseed: Earth School" The Movie!   
12242                               Urban farming housing community   
12490                                                        Catman   
12691                      Sage: Warrior Of The Fallen Gods (Movie)   
14029                                    The Greatest American Hero   
14702 

In [134]:
# total text length
clean_df['total_text_length'] = clean_df['blurb_char_length'] + clean_df['story_char_length'] + clean_df['rewards_char_length'] + clean_df['risks_char_length']
print(clean_df['total_text_length'].describe())

count   40,471.00
mean     5,928.02
std      4,792.04
min        195.00
25%      2,612.00
50%      4,641.00
75%      7,745.00
max     78,370.00
Name: total_text_length, dtype: float64


In [135]:
# check number of projects with 'goal_converted_USD' >= 1000000, 'story_text_length' < 1000,  percent_funded < 1
extreme_projects_3 = clean_df[(clean_df['goal_converted_USD'] >= 1000000) & (clean_df['story_char_length'] < 1000) & (clean_df['percent_funded'] < 1)]
print("Number of projects with 'goal_converted_USD' >= 1,000,000, 'story_char_length' < 1000,  percent_funded < 1:", extreme_projects_3.shape[0])

Number of projects with 'goal_converted_USD' >= 1,000,000, 'story_char_length' < 1000,  percent_funded < 1: 42


In [136]:
# check number of projects with 'goal_converted_USD' >= 1000000, all text attributes < 5000 and percent_funded < 1
extreme_projects_2 = clean_df[(clean_df['goal_converted_USD'] >= 1000000) & (clean_df['total_text_length'] < 5000) & (clean_df['percent_funded'] < 1)]
print("Number of projects with 'goal_converted_USD' >= 1,000,000, all text attributes < 5000 and percent_funded < 1:", extreme_projects_2.shape[0])

Number of projects with 'goal_converted_USD' >= 1,000,000, all text attributes < 5000 and percent_funded < 1: 99


In [137]:
# number of projects with a funding goal > $10,000,000
print("Number of projects with a funding goal >= $10,000,000:", (clean_df['goal_converted_USD'] >= 10_000_000).sum())

Number of projects with a funding goal >= $10,000,000: 16


In [94]:
# number of projects with a funding goal > $10,000,000 and were not at least 1% funded
print("Number of projects with a funding goal > $10,000,000 that were not at least 1% funded:", len(clean_df[(clean_df['goal_converted_USD'] >= 10_000_000) & (clean_df['percent_funded'] < 1)]))

Number of projects with a funding goal > $10,000,000 that were not at least 1% funded: 17


In [138]:
# number of projects with goal_converted_USD < 1000
low_goal_projects = clean_df[clean_df['goal_converted_USD'] < 1000]
print("Number of projects with goal_converted_USD < 1000:", low_goal_projects.shape[0])

Number of projects with goal_converted_USD < 1000: 6886


In [139]:
# number of projects with pledged_amount < 100
low_pledged_projects = clean_df[clean_df['pledged'] < 100]
print("Number of projects with pledged_amount < 100:", low_pledged_projects.shape[0])

Number of projects with pledged_amount < 100: 10300


In [140]:
# number of projects with pledged_amount = 0
zero_pledged_projects = clean_df[clean_df['pledged'] == 0]
print("Number of projects with pledged_amount = 0:", zero_pledged_projects.shape[0])

Number of projects with pledged_amount = 0: 2655


In [141]:
# number of projects with backers_count < 10
low_backer_projects = clean_df[clean_df['backers_count'] < 10]
print("Number of projects with backers_count < 10:", low_backer_projects.shape[0])

Number of projects with backers_count < 10: 15925


In [ ]:
# number of projects with less than 500 characters in all of their text attributes combined
short_text_projects = clean_df[clean_df['total_text_length'] < 500]
print("Number of projects with less than 500 characters in all text attributes combined:", short_text_projects.shape[0])

Number of projects with less than 500 characters in all text attributes combined: 116


In [ ]:
# number of projects with story length < 200 characters
print("Projects with story length < 200:", (clean_df['story_char_length'] < 200).sum())

# number of projects with story length < 200 characters and goal_converted_USD >= 100,000
short_story_high_goal = clean_df[
    (clean_df['story_char_length'] < 200) &
    (clean_df['goal_converted_USD'] >= 100_000)
]

print("Short story + high goal:", short_story_high_goal.shape[0])

Projects with story length < 200: 443
Short story + high goal: 20


In [144]:
# print a few examples
print(low_backer_projects[['name', 'backers_count', 'goal_converted_USD', 'pledged', 'percent_funded', 'state', 'urls_web_project']].head(10))

                                                          name  backers_count  \
5                             MikeyJ clothing brand fundraiser              1   
9                                            Daily Brew Coffee              1   
18                                  H2O - Underwater Portraits              2   
20                          Mobile barbecue & catering trailer              1   
23  "Flying" Carpets from Azerbaijan, Iran and Turkey/Anatolia              1   
25        Expi - look, find, decide (web, android and ios app)              2   
29                                   Delightfully Pure Candles              4   
32                                                  Baba Bongo              2   
36                         Just Another Reality Show: Season 3              8   
40      Save the Work of Douglas Davis, Pioneer of Digital Art              8   

    goal_converted_USD  pledged  percent_funded       state  \
5             1,980.35     1.00            0.

In [145]:
# number of project with an average pledge per backer of 3000
high_pledge_per_backer = clean_df[clean_df['usd_pledge_per_backer'] > 3000]
print("Number of projects with an average pledge per backer of > $3000:", high_pledge_per_backer.shape[0])

Number of projects with an average pledge per backer of > $3000: 50


In [146]:
# number of projects with an average pledge per backer greater than $3000
high_pledge_per_backer = clean_df[clean_df['usd_pledge_per_backer'] > 3000]
print("Number of projects with an average pledge per backer greater than $3000:", high_pledge_per_backer.shape[0])

Number of projects with an average pledge per backer greater than $3000: 50


In [ ]:
# Number of projects that collected more than 5,000% of their funding goal
highly_successful_projects = clean_df[clean_df['percent_funded'] > 6000]
print("Number of projects that collected more than 5,000% of their funding goal:", highly_successful_projects.shape[0])

Number of projects that collected more than 5,000% of their funding goal: 221


In [ ]:
# few examples of hihgly successful projects
print(highly_successful_projects[['name', 'goal_converted_USD', 'pledged', 'percent_funded', 'state', 'urls_web_project']].head(10))

                                                            name  \
74      To'ka Cloning Station - 3D printable Platforms and Minis   
127                                        Neokingdom - Full Set   
176                                         Midwest Side Stories   
338                                                 Tomb Raiders   
396                              Tiny Cats ❤️ Filler Enamel Pins   
433                                  Tons Of Music For A Buck!!!   
454          Oukitel WP300 Industrial-grade Modular Rugged Phone   
836   Kenophania: How A Song Is Created (Interactive Experience)   
1207                                Paradise University Season 5   
1253                                      TOWN OF FALLMOURE - II   

      goal_converted_USD    pledged  percent_funded       state  \
74                450.44  42,490.00       10,116.67  successful   
127               109.82  11,146.00       11,146.00  successful   
176                20.00   5,075.00       25,375.0

In [156]:
# remove campaigns with goal_converted_USD >= 10,000,000
print("Number of projects with goal_converted_USD >= 10,000,000:", (clean_df['goal_converted_USD'] >= 10000000).sum())
clean_df = clean_df[clean_df['goal_converted_USD'] < 10000000].reset_index(drop=True)
print("Removed campaigns with goal_converted_USD >= 10,000,000. New shape:", clean_df.shape)

Number of projects with goal_converted_USD >= 10,000,000: 16
Removed campaigns with goal_converted_USD >= 10,000,000. New shape: (40455, 79)


In [ ]:
# remove projects with goal >= $1,000,000 and story length <1000 and percent_funded < 1
print("Number of projects with goal >= $1,000,000 and story length <1000 and percent_funded < 1:", len(clean_df[(clean_df['goal_converted_USD'] >= 1000000) & (clean_df['story_char_length'] < 1000) & (clean_df['percent_funded'] < 1)]))
clean_df = clean_df[~((clean_df['goal_converted_USD'] >= 1000000) & (clean_df['story_char_length'] < 1000) & (clean_df['percent_funded'] < 1))].reset_index(drop=True)
print("Removed projects with goal >= $1,000,000 and story length <1000 and percent_funded < 1. New shape:", clean_df.shape)

Number of projects with goal >= $1,000,000 and story length <1000 and percent_funded < 1: 39
Removed projects with goal >= $1,000,000 and story length <1000 and percent_funded < 1. New shape: (40416, 79)


In [158]:
# remove projects with less than 500 characters in all text attributes combined
print("Number of projects with less than 500 characters in all text attributes combined:", (clean_df['total_text_length'] < 500).sum())
clean_df = clean_df[clean_df['total_text_length'] >= 500].reset_index(drop=True)
print("Removed projects with less than 500 characters in all text attributes combined. New shape:", clean_df.shape)

Number of projects with less than 500 characters in all text attributes combined: 115
Removed projects with less than 500 characters in all text attributes combined. New shape: (40301, 79)


In [161]:
# final shape of clean_df
print("Final shape of clean_df:", clean_df.shape)
# number of removed outliers
num_removed_outliers = 40669 - clean_df.shape[0]
print(f"Number of removed outliers: {num_removed_outliers}")

Final shape of clean_df: (40301, 79)
Number of removed outliers: 368


In [160]:
# save to new csv
clean_df.to_csv('final_dataset.csv', index=False)